# Data Manipulation using tidyr

In [15]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [21]:
## Long to Wide with pivot_wider

See: https://cengel.github.io/R-data-wrangling/tidyr.html

In [17]:
# We will be working a small subset of the data from the Stanford Open Policing Project. 
# It contains information about traffic stops in the state of Mississippi during January 2013 to mid-July of 2016
stops <- read_csv("../data/MS_trafficstops_bw_age.csv", show_col_types = FALSE)
head(stops)


id,stop_date,county_name,county_fips,police_department,driver_gender,driver_birthdate,driver_race,officer_id,driver_age,violation
<chr>,<date>,<chr>,<dbl>,<chr>,<chr>,<date>,<chr>,<chr>,<dbl>,<chr>
MS-2013-00001,2013-01-01,Jones,28067,Mississippi Highway Patrol,male,1950-06-14,Black,J042,63,Seat belt
MS-2013-00002,2013-01-01,Lauderdale,28075,Mississippi Highway Patrol,male,1967-04-06,Black,B026,46,Careless driving
MS-2013-00003,2013-01-01,Pike,28113,Mississippi Highway Patrol,male,1974-04-15,Black,M009,39,Speeding
MS-2013-00004,2013-01-01,Hancock,28045,Mississippi Highway Patrol,male,1981-03-23,White,K035,32,Speeding
MS-2013-00005,2013-01-01,Holmes,28051,Mississippi Highway Patrol,male,1992-08-03,White,D028,20,Speeding
MS-2013-00006,2013-01-01,Jackson,28059,Mississippi Highway Patrol,female,1960-05-02,White,K023,53,Speeding


# Long to Wide with pivot_wider

Now let’s see this in action. First, using dplyr, let’s create a data frame with the counts of different violations for each county

In [19]:
violations <- stops %>%
  count(county_name, violation, name = "n_violations")

violations

county_name,violation,n_violations
<chr>,<chr>,<int>
Adams,Breaks-Lights-etc,7
Adams,Careless driving,48
Adams,License-Permit-Insurance,118
Adams,Other or unknown,35
Adams,Seat belt,229
Adams,Speeding,505
Alcorn,Breaks-Lights-etc,62
Alcorn,Careless driving,100
Alcorn,License-Permit-Insurance,737


Now, to make this long data wide, we use pivot_wider from tidyr to turn the different violation types into columns, where each possible value of the violation variable receives its own column.

pivot_wider() takes three principal arguments:

1. the data
2. the names_from column variable whose values will become new column names.
3. the values_from column variable whose values will fill the new column cells.

We’ll use a pipe into pivot_wider so we can leave out the data argument

In [25]:
violations_wide <- violations %>%
  pivot_wider(names_from = violation, 
              values_from = n_violations) 

head(violations_wide)

county_name,Breaks-Lights-etc,Careless driving,License-Permit-Insurance,Other or unknown,Seat belt,Speeding
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
Adams,7,48,118,35,229,505
Alcorn,62,100,737,418,629,1399
Amite,47,86,370,143,336,1939
Attala,99,113,526,155,748,2562
Benton,3,9,73,26,29,74
Bolivar,57,139,1034,254,729,2313


# Wide to long with pivot_longer

What if we had the opposite problem, and wanted to go from a wide to long format? For that, we use pivot_longer, which will increase the number of rows and decrease the number of columns.

We are gathering the multiple violation columns and turn them into a pair of new variables. One variable includes the column names as values, and the other variable contains the values in each cell previously associated with the column names.

pivot_longer() takes four principal arguments:

1. the data
2. cols are the names of the columns we use to fill the a new values variable (or to drop).
3. the names_to a string specifying the name of the column to create from the data stored in the column names (cols)
4. the values_to which is also a string, specifying the name of the column to create from the data stored in cell values.
So, to go backwards from violations_wide, and exclude county_name from the long format, we would do the following:

In [33]:
violations_long <- violations_wide %>%
  pivot_longer(cols = -county_name,        # exclude column with county name
               names_to = "violation",     # name is a string!
               values_to = "n_violations")            # also a string

head(violations_long)

county_name,violation,n_violations
<chr>,<chr>,<int>
Adams,Breaks-Lights-etc,7
Adams,Careless driving,48
Adams,License-Permit-Insurance,118
Adams,Other or unknown,35
Adams,Seat belt,229
Adams,Speeding,505


See: https://tavareshugo.github.io/r-intro-tidyverse-gapminder/09-reshaping/index.html